# Notebook 03 — Tariff Pricing Agent

Translates demand forecasts into **dynamic per-kWh tariffs**:
- **Surge** when predicted utilization > 80%
- **Discount** when predicted utilization < 30%
- **Standard** otherwise

Compares against **₹15/kWh fixed baseline** (OP'26 evaluation metric).

**Metrics:** Revenue Gain %, Charger Utilization Rate, Off-Peak Uplift

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import config
from helpers.pricing import (
    apply_dynamic_pricing_df,
    revenue_gain_pct,
    off_peak_uplift,
    recommend_tariff,
)

sns.set_theme(style="whitegrid")

In [ ]:
# Load demand forecasts from Notebook 02
forecasts = pd.read_csv(config.PREDICTIONS_DIR / "demand_forecasts_test.csv")
forecasts["datetime"] = pd.to_datetime(forecasts["datetime"])
print("Forecast rows:", len(forecasts))
forecasts.head()

## 1. Dynamic Tariff Rules

| Tier | Condition | Price |
|------|-----------|-------|
| Surge | util > 80% OR congestion prob > 50% | ₹15 × 1.30 = **₹19.50** |
| Discount | util < 30% | ₹15 × 0.85 = **₹12.75** |
| Standard | else | **₹15.00** |

In [ ]:
# Apply pricing agent to each forecast row
pricing = apply_dynamic_pricing_df(
    forecasts,
    util_col="predicted_utilization",
    congestion_col="congestion_probability",
    volume_col="volume_kwh",
)

pricing["utilization"] = pricing["actual_utilization"]  # for monitoring metrics
print(pricing["pricing_tier"].value_counts())
pricing[["datetime", "station_id", "predicted_utilization", "pricing_tier", "fixed_price_inr", "dynamic_price_inr"]].head(10)

## 2. Revenue Comparison — Fixed vs Dynamic

In [ ]:
total_fixed = pricing["revenue_fixed_inr"].sum()
total_dynamic = pricing["revenue_dynamic_inr"].sum()
gain = revenue_gain_pct(total_dynamic, total_fixed)

print("=== Tariff Pricing Agent — Revenue Summary ===")
print(f"Fixed baseline revenue (₹15/kWh):   ₹{total_fixed:,.0f}")
print(f"Dynamic pricing revenue:              ₹{total_dynamic:,.0f}")
print(f"Revenue Gain %:                       {gain:.2f}%")

In [ ]:
# Visual: revenue by pricing tier
tier_rev = pricing.groupby("pricing_tier").agg(
    revenue_dynamic=("revenue_dynamic_inr", "sum"),
    revenue_fixed=("revenue_fixed_inr", "sum"),
    avg_util=("actual_utilization", "mean"),
    rows=("station_id", "count"),
)
display(tier_rev)

fig, ax = plt.subplots(figsize=(7, 4))
x = tier_rev.index
width = 0.35
ax.bar(x, tier_rev["revenue_fixed"], width, label="Fixed ₹15/kWh")
ax.bar([i + width for i in range(len(x))], tier_rev["revenue_dynamic"], width, label="Dynamic")
ax.set_xticks([i + width/2 for i in range(len(x))])
ax.set_xticklabels(x)
ax.set_title("Revenue by Pricing Tier (Test Period)")
ax.set_ylabel("Revenue (INR)")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Utilization Before vs After (simulated demand response)

We simulate a simple **price elasticity** on volume:
- Surge → 5% volume reduction (discourages peak usage)
- Discount → 8% volume increase (attracts off-peak sessions)

> **Assumption (document for submission):** This is a demand-response proxy, not a causal claim.

In [ ]:
ELASTICITY = {"surge": 0.95, "discount": 1.08, "standard": 1.00}

pricing["volume_after"] = pricing.apply(
    lambda r: r["volume_kwh"] * ELASTICITY[r["pricing_tier"]], axis=1
)

# Utilization after: scale by volume change / capacity proxy
pricing["utilization_after"] = (
    pricing["actual_utilization"] * (pricing["volume_after"] / pricing["volume_kwh"].replace(0, np.nan))
).clip(0, 1).fillna(pricing["actual_utilization"])

util_before = pricing["actual_utilization"].mean()
util_after = pricing["utilization_after"].mean()
print(f"Mean utilization BEFORE dynamic pricing: {util_before:.3f}")
print(f"Mean utilization AFTER (simulated):      {util_after:.3f}")

## 4. Off-Peak Uplift (OP'26 Metric)

In [ ]:
uplift = off_peak_uplift(pricing, util_col="actual_utilization")
print(f"Off-Peak Uplift (discount periods, util < 30%): {uplift:.2f}%")

off_peak = pricing[pricing["actual_utilization"] < config.DISCOUNT_UTIL_THRESHOLD]
print(f"Off-peak buckets: {len(off_peak)} | Discount buckets: {(off_peak['pricing_tier']=='discount').sum()}")

## 5. Hourly Tariff Heatmap (presentation visualization)

In [ ]:
pricing["hour"] = pricing["datetime"].dt.hour
heatmap_data = pricing.groupby(["hour", "pricing_tier"]).size().unstack(fill_value=0)

plt.figure(figsize=(10, 4))
sns.heatmap(heatmap_data.T, cmap="YlOrRd", annot=False)
plt.title("Pricing Tier Frequency by Hour of Day")
plt.xlabel("Hour")
plt.ylabel("Pricing Tier")
plt.tight_layout()
plt.show()

## 6. Save Tariff Recommendations & Summary

In [ ]:
pricing.to_csv(config.TARIFFS_DIR / "tariff_recommendations_test.csv", index=False)

summary = pd.DataFrame([{
    "fixed_baseline_inr_per_kwh": config.FIXED_BASELINE_PRICE_INR,
    "surge_multiplier": config.SURGE_MULTIPLIER,
    "discount_multiplier": config.DISCOUNT_MULTIPLIER,
    "total_revenue_fixed_inr": total_fixed,
    "total_revenue_dynamic_inr": total_dynamic,
    "revenue_gain_pct": gain,
    "mean_utilization_before": util_before,
    "mean_utilization_after_simulated": util_after,
    "off_peak_uplift_pct": uplift,
}])
summary.to_csv(config.REPORTS_DIR / "tariff_pricing_summary.csv", index=False)
print("Saved tariff outputs to", config.TARIFFS_DIR)